In [ ]:
import os
import gc
import math
import random

import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from joblib import Parallel, delayed
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, hsv_to_rgb

import cfospy

In [ ]:
def calc_CT_norm(CT_df, th):
    """Z-score normalize per region and compute per-CT means across 4 days."""
    CTs = np.arange(0, 24, 4)  # [0, 4, ..., 20]
    num_samp = 6               # samples per time point

    # Filter by BH.Q threshold
    df_i = CT_df[CT_df["BH.Q"] < th]
    ids = df_i["id"].tolist()

    print(f"Number of regions (BH.Q < {th}): {len(df_i)}")

    # Extract measurement matrix (columns starting with "CT")
    ct_cols = df_i.columns[df_i.columns.str.startswith("CT")]
    df_all = df_i[ct_cols].to_numpy()

    # Per-row z-score normalization
    mean = df_all.mean(axis=1, keepdims=True)
    std = df_all.std(axis=1, keepdims=True)
    df_i = (df_all - mean) / std

    # Aggregate 4 days for each CT by averaging the corresponding 6-sample blocks
    CT_ms = np.zeros((df_i.shape[0], len(CTs)))
    CT_sds = np.zeros_like(CT_ms)

    block = num_samp
    day_stride = block * 6  # 36 columns per day

    for CT_i in range(len(CTs)):
        start = CT_i * block
        end = (CT_i + 1) * block

        CT_li = (
            df_i[:, start:end]
            + df_i[:, start + day_stride:end + day_stride]
            + df_i[:, start + 2 * day_stride:end + 2 * day_stride]
            + df_i[:, start + 3 * day_stride:end + 3 * day_stride]
        ) / 4

        CT_ms[:, CT_i] = CT_li.mean(axis=1)

    return ids, df_i, CT_ms


def max_corr_fig(CT_df, vnorm, a, exp, sample, outdir, fig=True, cvd=False):
    """Find phase with max correlation and plot."""
    bs = np.linspace(0, 24, 144)
    peakt = np.array(CT_df["LAG"])

    corrs = []
    for b in bs:
        cos_p = a * np.sqrt(2) * np.cos(2 * np.pi * (peakt - b) / 24)
        corr, _ = pearsonr(vnorm, cos_p)
        corrs.append(corr)

    bc = bs[np.argmax(corrs)]
    cmax = np.max(corrs)

    if fig == True:
        if cvd == True:
            ph_li = CT_df["LAG"] / 24
            rgba_float = cmap_icefire(np.array(ph_li))
            colors = rgba_float[:, :3]
        else:
            ph_li = 1 - CT_df["LAG"] / 24
            ph_li = [(h + 1/3 - 1) if (h + 1/3) > 1 else (h + 1/3) for h in ph_li]
            colors = [hsv_to_rgb([h, 1, 1]) for h in ph_li]
    
        yl = 2.5
        plt.figure(figsize=(6, 4))
        plt.scatter(CT_df["LAG"], vnorm, c=colors)
        plt.plot(CT_df["LAG"], a * np.sqrt(2) * np.cos(2 * np.pi * (peakt - bc) / 24), color="k")
        plt.xticks(np.arange(0, 28, 4))
        plt.yticks(np.arange(-yl, yl + 0.1, yl))
        plt.xlim(0, 24)
        plt.ylim(-yl, yl)
    
        ax = plt.gca()
        ax.axes.xaxis.set_ticklabels([])
        ax.axes.yaxis.set_ticklabels([])
    
        os.makedirs(outdir, exist_ok=True)
        plt.savefig(os.path.join(outdir, f"level_peak_time_cr_test_{exp}_{sample}.SVG"))
        plt.show()

    print(bc)
    return cmax, bc


def calc_score(sample_names, predicted, fig_op=False):
    """Compute timetable score (sum of min absolute differences under 24h wrap)."""
    # True labels from sample names like "CT0_01" → [0, 4, 8, ...]
    ans = [int(s.split("_")[0][2:]) for s in sample_names]
    ans2 = [(a - 24) if a > 20 else a for a in ans]

    # Consider wrap-around (pred±24) and choose min absolute difference
    predicted = np.asarray(predicted, dtype=np.float32)
    ans2 = np.asarray(ans2, dtype=np.float32)

    diff = np.zeros((3, len(sample_names)), dtype=np.float32)
    diff[0] = np.abs(predicted)
    diff[1] = np.abs(predicted + 24)
    diff[2] = np.abs(predicted - 24)

    diff_a = np.abs(diff - ans2)  # shape (3, N)

    min_diff = np.min(diff_a, axis=0)
    score = float(np.sum(min_diff))
    sortin = np.argmin(diff_a, axis=0)

    predicted_t = [diff[s, n] for n, s in enumerate(sortin)]

    print(f"Score (sum of min diffs): {score}")

    if fig_op == True:
        plt.figure(figsize=(10, 6))
        x = np.arange(len(sample_names))
        plt.scatter(x, ans2, s=5)
        plt.scatter(x, predicted_t, s=6)
        plt.plot(ans2, label="True")
        plt.plot(predicted_t, label="Predicted")
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0)
        plt.yticks(np.arange(0, 28, 4))
        plt.ylim(0, )
        ax = plt.gca()
        ax.axes.xaxis.set_ticklabels([])
        ax.axes.yaxis.set_ticklabels([])
        plt.show()

    return score, min_diff
    

def diff_plot(min_diff, fig_op=False):
    """Plot mean±SEM of phase prediction errors at each CT bin."""
    CT_li = np.arange(0, 24, 4)
    diff_mean = np.zeros(len(CT_li), dtype=np.float32)
    diff_SD = np.zeros(len(CT_li), dtype=np.float32)

    for i, CT in enumerate(CT_li):
        v = min_diff[i:i+6]
        v = np.append(v, min_diff[i+24:i+30])
        if len(min_diff) >= 73:
            v = np.append(v, min_diff[i+48:i+54])
            v = np.append(v, min_diff[i+72:i+78])
        print(v)
        diff_mean[i] = np.mean(v)
        diff_SD[i] = np.std(v) / np.sqrt(len(v))

    print(f"mean {np.mean(diff_mean)}")
    print(f"SD {np.mean(diff_SD)}")

    if fig_op == True:
        plt.errorbar(CT_li, diff_mean, yerr=diff_SD, capsize=5)
        plt.scatter(CT_li, diff_mean, s=15)
        plt.xticks(np.arange(0, 24, 4))
        plt.ylim(0, 2.0)
        plt.show()

    return float(np.mean(diff_mean))


def calc_cmax_bc(peakt, vnorm):
    """Return max correlation (cmax) and phase (bc) between vnorm and a unit cosine."""
    bs = np.linspace(0, 24, 144)
    corrs = []
    for b in bs:
        cos_p = np.sqrt(2) * np.cos(2 * np.pi * (peakt - b) / 24)
        corr, _ = pearsonr(vnorm, cos_p)
        corrs.append(corr)
    bc = bs[np.argmax(corrs)]
    cmax = np.max(corrs)
    return cmax, bc


def cont_pro_dis(N, mn, peakt, di_li):
    """Generate N continuous profiles with noise and collect cmax."""
    cmaxs_c = np.zeros(N)
    for n in range(N):
        cr_li = np.array([
            np.sqrt(2) * np.cos(2 * np.pi * peakt[i] / 24) + mn * random.choice(di_li)
            for i in range(len(peakt))
        ])
        cmax, _ = calc_cmax_bc(peakt, cr_li)
        cmaxs_c[n] = cmax
    return cmaxs_c


def job_cont_dis(n, mn, peakt, di_li, cmaxs_c):
    """Worker: compute cmax for one noisy profile and store into cmaxs_c[n]."""
    cr_li = np.array([
        np.sqrt(2) * np.cos(2 * np.pi * peakt[i] / 24) + mn * random.choice(di_li)
        for i in range(len(peakt))
    ])
    cmax, _ = calc_cmax_bc(peakt, cr_li)
    cmaxs_c[n] = cmax


def multi_cont_dis(N, mn, peakt, di_li, ncore):
    """Parallel version of cont_pro_dis using shared memory array."""
    cmaxs_c = np.zeros(N)
    Parallel(n_jobs=ncore, require="sharedmem")(
        [delayed(job_cont_dis)(n, mn, peakt, di_li, cmaxs_c) for n in range(N)]
    )
    return cmaxs_c


def job_rand_dis(n, mn, peakt, vnorm, cmaxs_r):
    """Worker: shuffle-scaled vnorm and store cmax into cmaxs_r[n]."""
    sh_norm = mn * np.copy(vnorm)
    np.random.shuffle(sh_norm)
    cmax, _ = calc_cmax_bc(peakt, sh_norm)
    cmaxs_r[n] = cmax


def multi_rand_dis(N, mn, peakt, vnorm, ncore):
    """Parallel version of random-shuffle test using shared memory array."""
    cmaxs_r = np.zeros(N)
    Parallel(n_jobs=ncore, require="sharedmem")(
        [delayed(job_rand_dis)(n, mn, peakt, vnorm, cmaxs_r) for n in range(N)]
    )
    return cmaxs_r


def get_pv_c(cmax_i, cmaxs):
    """One-sided p-value: P(C ≤ cmax_i) from empirical distribution."""
    cmaxs = np.sort(cmaxs)
    minx = np.argmin(np.abs(cmaxs - cmax_i))
    if cmaxs[minx] >= cmax_i:
        up_p = minx / len(cmaxs)
    else:
        up_p = (minx + 1) / len(cmaxs)
    return up_p


def get_pv_r(cmax_i, cmaxs):
    """One-sided p-value: P(C ≥ cmax_i) from empirical distribution."""
    cmaxs = np.sort(cmaxs)
    minx = np.argmin(np.abs(cmaxs - cmax_i))
    if cmaxs[minx] >= cmax_i:
        up_p = (len(cmaxs) - minx) / len(cmaxs)
    else:
        up_p = (len(cmaxs) - (minx + 1)) / len(cmaxs)
    return up_p

In [ ]:
src = "/path/to/source_dir"
dst = "/path/to/output_dir"

cfos_dir = os.path.join(src, "circadian_1st", "circadian_1st_Reconst")
savedir = os.path.join(dst, "cfos_app")

In [ ]:
# Collect unique sample names

CT_li = np.arange(0, 48, 4)  # circadian time points (CT0–44, every 4 h)
sample_ids = np.arange(1, 7, 1)

reconsts = os.listdir(cfos_dir)
sample_names = []

for CT in CT_li:
    for sample_id in sample_ids:
        sample = f"CT{CT}_{str(sample_id).zfill(2)}"
        for reconst in reconsts:
            if sample in reconst:
                sample_names.append(sample)

print(len(sample_names))

In [ ]:
# Load atlas data
rdir = os.path.join(src, "CUBIC_R_atlas_ver5")

vx = 50
ca = cfospy.analysis.read_atlas_data(rdir, vx)
print(f"{len(ca.ID_all)} regions")

In [ ]:
# Read rhythmicity data

cos_dir = os.path.join(src, "cos_results")
res = "cos.cell_count_ratio_1st2nd_small_ai_fpr0.5.csv"

# Read cosinor test results
ct_path = os.path.join(cos_dir, res)
CT_df = pd.read_csv(ct_path)

CT_df

In [ ]:
# Homogenize distribution across circadian phases

num = 10          # max number of regions per phase bin
interval = 1.0    # bin width (in hours)
total = int(24 / interval)

counts = []
CT_all = []

for i in range(total):
    phase_min = i * interval
    phase_max = (i + 1) * interval

    # Select rows within this phase interval
    CT_ph = CT_df[(CT_df["LAG"] >= phase_min) & (CT_df["LAG"] < phase_max)]
    count = len(CT_ph)
    counts.append(count)
    print(f"{phase_min:.1f} h: {count} regions")

    # Limit the number of entries per bin by smallest BH.Q
    CT_ph = CT_ph.sort_values("BH.Q").iloc[:num]
    CT_all.append(CT_ph)

# Combine and sort by phase
CT_df = pd.concat(CT_all, axis=0)
print(f"Total after homogenization: {len(CT_df)} regions")

CT_df = CT_df.sort_values("LAG")

In [ ]:
# Plot histogram of phase distribution
figdir = os.path.join(savedir, "figure_timetable")
os.makedirs(figdir, exist_ok=True)

print(f"Number of regions: {len(CT_df)}")

plt.figure(figsize=(6, 4))
plt.hist(CT_df["LAG"], bins=12, color="skyblue")
plt.xticks(np.arange(0, 28, 4))
plt.yticks(np.arange(0, 100, 20))

ax = plt.gca()
ax.axes.xaxis.set_ticklabels([])
ax.axes.yaxis.set_ticklabels([])

outfile = os.path.join(figdir, "count_ratio_phase_hist_all.SVG")
plt.savefig(outfile)
plt.show()

In [ ]:
# Filter phase data by experimental batch and day
th = 0.1
ids, df_i, CT_norm = calc_CT_norm(CT_df, th)  # all regions passing threshold

CT_norm.shape

In [ ]:
# Output of timetable plots
name = "1st_day1"
yl = 2.5

for i in range(6):
    ph_li = 1 - CT_df["LAG"] / 24
    ph_li = [(h + 1/3 - 1) if (h + 1/3) > 1 else (h + 1/3) for h in ph_li]
    colors = [hsv_to_rgb([h, 1, 1]) for h in ph_li]
    
    # Color-vision-friendly option:
    # rgba_float = cmap_icefire(np.array(ph_li))
    # colors = rgba_float[:, :3] 

    plt.figure(figsize=(6, 4))
    plt.scatter(CT_df["LAG"], CT_norm[:, i], color=colors)
    plt.xticks(np.arange(0, 28, 4))
    plt.yticks(np.arange(-yl, yl + 0.1, yl))
    plt.xlim(0, 24)
    plt.ylim(-yl, yl)

    ax = plt.gca()
    ax.axes.xaxis.set_ticklabels([])
    ax.axes.yaxis.set_ticklabels([])

    os.makedirs(figdir, exist_ok=True)
    filename = f"level_peak_time_cr_{name}_CT{i*4}_homo.SVG"
    plt.savefig(os.path.join(figdir, filename))
    plt.show()

In [ ]:
# Colormap optimized for color vision deficiency

color_points_mod = [
    (0.00, (1.00, 0.45, 0.15)),  # orange
    (0.25, (1.00, 1.00, 0.40)),  # yellowish
    (0.50, (0.60, 0.90, 0.90)),  # light cyan
    (0.75, (0.15, 0.75, 1.00)),  # cyan-blue
    (1.00, (0.20, 0.05, 0.85))   # blue-violet
]

cmap_icefire = LinearSegmentedColormap.from_list(
    "erdc_icefire_darkred",
    color_points_mod,
    N=256
)

fig, ax = plt.subplots(figsize=(8, 1.0))
grad = np.linspace(0, 1, 1200)[None, :]
ax.imshow(grad, aspect='auto', cmap=cmap_icefire, extent=[0, 24, 0, 1])
ax.set_yticks([])
ax.set_xlim(0, 24)
ax.set_xlabel("CT (h)")
ax.set_xticks([0, 6, 12, 18, 24])
plt.tight_layout()

In [ ]:
# Predict brain time of each individual using the remaining 143 samples
exps = ["1st", "2nd"]
offsets = [0, 48]
op = "_ai_fpr0.5"
a = 1

predicted = []
phase_li = []

for k, exp in enumerate(exps):
    offset = offsets[k]
    
    for i, sample in enumerate(sample_names):
        sample_names_p = [j for j in sample_names if j != sample]

        # Define target sample name with batch offset
        sample2 = f"CT{int(sample.split('_')[0][2:]) + offset}_{sample.split('_')[1]}"
        print(sample2)

        # Exclude target sample for prediction
        CT_df_re = CT_df.drop(sample2, axis=1)

        # Select only columns that start with "CT"
        ct_cols = CT_df_re.columns[CT_df_re.columns.str.startswith("CT")]
        ct_vals = CT_df_re[ct_cols].to_numpy()
        
        # Mean and SD across the remaining samples
        mi = np.mean(ct_vals, axis=1)
        SDi = np.std(ct_vals, axis=1)

        # Filter regions (only small IDs)
        df = CT_df.copy()
        small_IDs = [ind for ind in ca.df_allen["ID"] if ca.smallID_q(ind)]
        df = df[df["id"].isin(small_IDs)]

        # Reorder rows to match CT_df_re
        sort_id = CT_df_re["id"].tolist()
        df = df.set_index("id").loc[sort_id].reset_index()

        # Normalize target sample by mean and SD
        vnorm = np.array([(df[sample2].iloc[i] - mi.iloc[i]) / SDi.iloc[i] for i in range(len(df))])
        print(len(vnorm))

        # Phase estimation
        cmax, phase = max_corr_fig(CT_df, vnorm, a, exp, sample, figdir, fig=False)
        phase_li.append(phase)
        predicted.append(phase)

In [ ]:
# Calculate timetable prediction scores
score, min_diff = calc_score(sample_names+sample_names, predicted, fig_op=True)
diff_plot(min_diff, fig_op=True)

In [ ]:
# Projection mapping of results

CT_li = np.array([int(i.split("_")[0][2:]) for i in sample_names + sample_names])
ph_li = 1 - (CT_li % 24) / 24
ph_li = [(h + 1/3 - 1) if (h + 1/3) > 1 else (h + 1/3) for h in ph_li]

x = np.array(amp_li) * np.cos(rads)
y = np.array(amp_li) * np.sin(rads)

plt.figure(figsize=(6, 6))
for i in range(len(x)):
    color = hsv_to_rgb([ph_li[i], 1, 1])
    if i % 6 == 0 and i < 36:
        plt.scatter(x[i], y[i], s=5, color=color, edgecolor=None, label=CT_li[i])
    else:
        plt.scatter(x[i], y[i], s=5, color=color, edgecolor=None)
        
plt.xlim()
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", borderaxespad=0)
os.makedirs(figdir, exist_ok=True)
plt.savefig(os.path.join(figdir, "timetable_projection_annot.SVG"))
plt.show()

In [ ]:
# Timetable method under different FDR thresholds
FDR_li = [0.1, 0.05, 0.01, 0.005, 0.001, 0.0005, 0.0001, 0.00005, 0.00001, 0.000005, 0.000001]

scores = []
mean_diff_li = []
region_nums = []

for FDR_th in FDR_li:
    # Filter regions by BH.Q and sort by phase
    CT_df2 = CT_df[CT_df["BH.Q"] < FDR_th].sort_values("LAG")
    region_nums.append(len(CT_df2))

    # Normalize per CT
    ids, df_i, CT_norm = calc_CT_norm(CT_df2, FDR_th)

    predicted = []
    phase_li = []

    for k, exp in enumerate(exps):
        offset = offsets[k]

        for i, sample in enumerate(sample_names):
            # Define target sample name with batch offset
            sample2 = f"CT{int(sample.split('_')[0][2:]) + offset}_{sample.split('_')[1]}"
            print(sample2)

            # Exclude target sample for prediction
            CT_df_re = CT_df2.drop(sample2, axis=1)
    
            # Select only columns that start with "CT"
            ct_cols = CT_df_re.columns[CT_df_re.columns.str.startswith("CT")]
            ct_vals = CT_df_re[ct_cols].to_numpy()
            
            # Mean and SD across the remaining samples
            mi = np.mean(ct_vals, axis=1)
            SDi = np.std(ct_vals, axis=1)

            # Filter regions (only small IDs)
            df = CT_df2.copy()
            small_IDs = [ind for ind in ca.df_allen["ID"] if ca.smallID_q(ind)]
            df = df[df["id"].isin(small_IDs)]

            # Reorder rows to match CT_df_re
            sort_id = CT_df_re["id"].tolist()
            df = df.set_index("id").loc[sort_id].reset_index()

            # Z-score for the held-out sample
            vnorm = [(df[sample2].iloc[i] - mi.iloc[i]) / SDi.iloc[i] for i in range(len(df))]
            vnorm = np.nan_to_num(vnorm)
            vnorm = vnorm[vnorm != 0]
            CT_df_d = CT_df2.iloc[np.where(vnorm != 0)[0]]
            vnorm = np.array(vnorm)

            # Phase estimation
            cmax, phase = max_corr_fig(CT_df_d, vnorm, a, exp, sample, figdir, fig=False)
            phase_li.append(phase)
            predicted.append(phase)

    print(FDR_th)
    score, min_diff = calc_score(sample_names + sample_names, predicted)
    scores.append(score)

    mean_diff = diff_plot(min_diff)
    mean_diff_li.append(mean_diff)

plt.plot(np.log10(FDR_li), scores)
plt.xlabel("log10(FDR threshold)", fontsize=15)
plt.ylabel("Score", fontsize=15)
plt.show()

In [ ]:
# Plot FDR threshold vs score, mean difference, and number of regions
fs = 15

# Score plot
plt.plot(np.log10(FDR_li)[::-1], scores[::-1])
plt.scatter(np.log10(FDR_li)[::-1], scores[::-1])
plt.xlabel("log10(FDR threshold)", fontsize=fs)
plt.ylabel("Score", fontsize=fs)
plt.savefig(os.path.join(figdir, "timetable_FDR_score.SVG"))
plt.show()

# Mean difference plot
plt.plot(np.log10(FDR_li)[::-1], mean_diff_li[::-1])
plt.scatter(np.log10(FDR_li)[::-1], mean_diff_li[::-1])
plt.xlabel("log10(FDR threshold)", fontsize=fs)
plt.ylabel("Mean difference", fontsize=fs)
plt.yticks(np.arange(0, 3, 0.5))
plt.ylim(0,)
ax = plt.gca()
ax.axes.xaxis.set_ticklabels([])
ax.axes.yaxis.set_ticklabels([])
plt.savefig(os.path.join(figdir, "timetable_FDR_mean_diff.SVG"))
plt.show()

# Region count plot
plt.plot(np.log10(FDR_li)[::-1], region_nums[::-1])
plt.scatter(np.log10(FDR_li)[::-1], region_nums[::-1])
plt.xlabel("log10(FDR threshold)", fontsize=fs)
plt.ylabel("Number of regions", fontsize=fs)
ax = plt.gca()
ax.axes.xaxis.set_ticklabels([])
ax.axes.yaxis.set_ticklabels([])
plt.ylim(0,)
plt.savefig(os.path.join(figdir, "timetable_FDR_region_num.SVG"))
plt.show()

In [ ]:
# Obtain c_th by sample → average correlation threshold

N = 10000
ncore = 30
mn = 1
cmax_i_li = np.arange(0, 1.0, 0.01)
bs = np.linspace(0, 24, 144)
FDR_li = [0.0001]

# Random sampling (Nr samples across batches)
exp_rs = []
sample_names_rs = []
Nr = 10

ct_cols = CT_df.columns[CT_df.columns.str.startswith("CT")]

for i in range(Nr):
    sample_r = random.choice(list(ct_cols))
    print(sample_r)
    if int(sample_r.split("_")[0][2:]) > 44:
        exp_rs.append("2nd")
        sample_names_rs.append(f"CT{int(sample_r.split('_')[0][2:]) - 48}_{sample_r.split('_')[1]}")
    else:
        exp_rs.append("1st")
        sample_names_rs.append(sample_r)

print(exp_rs)
print(sample_names_rs)

# Calculate average threshold
region_nums = []
for FDR_th in FDR_li:
    CT_df2 = CT_df[CT_df["BH.Q"] < FDR_th].sort_values("LAG")
    region_nums.append(len(CT_df2))
    print(len(CT_df2))

    c_ths = []
    predicted = []
    phase_li = []
    cmax_li = []

    for k, (exp, sample) in enumerate(zip(exp_rs, sample_names_rs)):
        offset = 48 if exp == "2nd" else 0

        sample2 = f"CT{int(sample.split('_')[0][2:]) + offset}_{sample.split('_')[1]}"
        print(sample2)
        
        CT_df_re = CT_df2.drop(sample2, axis=1)

        ct_cols = CT_df_re.columns[CT_df_re.columns.str.startswith("CT")]
        ct_vals = CT_df_re[ct_cols].to_numpy()
        
        mi = np.mean(ct_vals, axis=1)
        SDi = np.std(ct_vals, axis=1)

        df = CT_df2.copy()
        small_IDs = [ind for ind in ca.df_allen["ID"] if ca.smallID_q(ind)]
        df = df[df["id"].isin(small_IDs)]

        sort_id = CT_df_re["id"].tolist()
        df = df.set_index("id").loc[sort_id].reset_index()

        vnorm = [(df[sample2].iloc[i] - mi.iloc[i]) / SDi.iloc[i] for i in range(len(df))]
        vnorm = np.nan_to_num(vnorm)
        vnorm = vnorm[vnorm != 0]
        CT_df_d = CT_df2.iloc[np.where(vnorm != 0)[0]]
        vnorm = np.array(vnorm)

        peakt = np.array(CT_df_d["LAG"])

        corrs = []
        for b in bs:
            cos_p = np.sqrt(2) * np.cos(2 * np.pi * (peakt - b) / 24)
            corr, _ = pearsonr(vnorm, cos_p)
            corrs.append(corr)

        bc = bs[np.argmax(corrs)]
        cmax = np.max(corrs)
        cmax_li.append(cmax)

        di_li = vnorm - np.sqrt(2) * np.cos(2 * np.pi * (peakt - bc) / 24)

        cmaxs_c = multi_cont_dis(N, mn, peakt, di_li, ncore)
        cmaxs_r = multi_rand_dis(N, mn, peakt, vnorm, ncore)

        Sc_li = np.zeros(len(cmax_i_li), dtype="float32")
        Sr_li = np.zeros(len(cmax_i_li), dtype="float32")

        for j, cmax_i in enumerate(cmax_i_li):
            Pr = get_pv_r(cmax_i, cmaxs_r)
            Pc = get_pv_c(cmax_i, cmaxs_c)
            Sc_li[j] = 1 - Pc  # sensitivity
            Sr_li[j] = 1 - Pr  # specificity

        min_in = np.argmin(np.abs(Sc_li - Sr_li))
        c_th = cmax_i_li[min_in]

        outdir = os.path.join(savedir, f"timetable2/timetable_corrth_fdr{FDR_th}", exp)
        os.makedirs(outdir, exist_ok=True)
        outfile = os.path.join(outdir, f"{sample}.txt")
        with open(outfile, "w") as f:
            f.write(f"{c_th}\n")

        c_ths.append(c_th)
        print(f"fdr{FDR_th}_{exp}_{sample}_c_th: {c_th}")

    c_th_ave = np.mean(c_ths)
    print(f"fdr{FDR_th}_c_th_average: {c_th_ave}")

In [ ]:
# Auto-evaluate sensitivity and specificity for each FDR threshold
FDR_th_li = [0.1, 0.05, 0.01, 0.005, 0.001, 0.0005, 0.0001, 0.00005, 0.00001, 0.000005, 0.000001]
c_th_ave_li = [0.089, 0.24, 0.310, 0.324, 0.442, 0.483, 0.590, 0.624, 0.741, 0.867, 0.976]

sens = []
spec = []

for FDR_th, c_th_ave in zip(FDR_th_li, c_th_ave_li):
    plus_c = 0
    random_c = 0

    CT_df2 = CT_df[CT_df["BH.Q"] < FDR_th].sort_values("LAG")

    for k, exp in enumerate(exps):
        offset = offsets[k]
        for i, sample in enumerate(sample_names):
            sample2 = f"CT{int(sample.split('_')[0][2:]) + offset}_{sample.split('_')[1]}"
            print(sample2)

            CT_df_re = CT_df2.drop(sample2, axis=1)
            
            ct_cols = CT_df_re.columns[CT_df_re.columns.str.startswith("CT")]
            ct_vals = CT_df_re[ct_cols].to_numpy()
            
            mi = np.mean(ct_vals, axis=1)
            SDi = np.std(ct_vals, axis=1)

            df = CT_df2.copy()
            small_IDs = [ind for ind in ca.df_allen["ID"] if ca.smallID_q(ind)]
            df = df[df["id"].isin(small_IDs)]

            sort_id = CT_df_re["id"].tolist()
            df = df.set_index("id").loc[sort_id].reset_index()

            vnorm = [(df[sample2].iloc[i] - mi.iloc[i]) / SDi.iloc[i] for i in range(len(df))]
            vnorm = np.nan_to_num(vnorm)
            vnorm = vnorm[vnorm != 0]
            CT_df_d = CT_df2.iloc[np.where(vnorm != 0)[0]]
            vnorm = np.array(vnorm)

            peakt = np.array(CT_df_d["LAG"])

            corrs = []
            for b in bs:
                cos_p = np.sqrt(2) * np.cos(2 * np.pi * (peakt - b) / 24)
                corr, _ = pearsonr(vnorm, cos_p)
                corrs.append(corr)

            bc = bs[np.argmax(corrs)]
            cmax = np.max(corrs)
            print(cmax)

            if cmax > c_th_ave:
                plus_c += 1

            vnorm_r = vnorm.copy()
            np.random.shuffle(vnorm_r)
            corrs_r = []
            for b in bs:
                cos_p = np.sqrt(2) * np.cos(2 * np.pi * (peakt - b) / 24)
                corr, _ = pearsonr(vnorm_r, cos_p)
                corrs_r.append(corr)
            bc_r = bs[np.argmax(corrs_r)]
            cmax_r = np.max(corrs_r)
            print("random", cmax_r)

            if cmax_r < c_th_ave:
                random_c += 1

    per_sens = (plus_c / (len(sample_names) * 2)) * 100
    per_spec = (random_c / (len(sample_names) * 2)) * 100
    print(f"FDR{FDR_th}, sensitivity:{per_sens}%")
    print(f"FDR{FDR_th}, specificity:{per_spec}%")

    sens.append(per_sens)
    spec.append(per_spec)

In [ ]:
# Plot sensitivity and specificity
fs = 15

# Sensitivity
plt.figure(figsize=(5, 5))
plt.plot(np.log10(FDR_th_li)[::-1], sens[::-1])
plt.scatter(np.log10(FDR_th_li)[::-1], sens[::-1])
plt.xlabel("log10(FDR threshold)", fontsize=fs)
plt.ylabel("Sensitivity (%)", fontsize=fs)
plt.xticks(np.arange(-6, -0.9, 1))
ax = plt.gca()
ax.axes.xaxis.set_ticklabels([])
ax.axes.yaxis.set_ticklabels([])
plt.ylim(0, 105)
plt.savefig(os.path.join(figdir, "timetable_FDR_sensitivity.SVG"))
plt.show()

# Specificity
plt.figure(figsize=(5, 5))
plt.plot(np.log10(FDR_th_li)[::-1], spec[::-1])
plt.scatter(np.log10(FDR_th_li)[::-1], spec[::-1])
plt.xlabel("log10(FDR threshold)", fontsize=fs)
plt.ylabel("Specificity (%)", fontsize=fs)
plt.xticks(np.arange(-6, -0.9, 1))
ax = plt.gca()
ax.axes.xaxis.set_ticklabels([])
ax.axes.yaxis.set_ticklabels([])
plt.ylim(0, 105)
plt.savefig(os.path.join(figdir, "timetable_FDR_specificity.SVG"))
plt.show()

In [ ]:
# % of c_th > ... → sensitivity/specificity by CT bins
FDR_th = 0.1
c_th_ave = 0.089

CT_df2 = CT_df[CT_df["BH.Q"] < FDR_th].sort_values("LAG")

CT_c_c = {"0": 0, "4": 0, "8": 0, "12": 0, "16": 0, "20": 0}
CT_r_c = {"0": 0, "4": 0, "8": 0, "12": 0, "16": 0, "20": 0}

for k, exp in enumerate(exps):
    offset = offsets[k]
    for i, sample in enumerate(sample_names):
        sample2 = f"CT{int(sample.split('_')[0][2:]) + offset}_{sample.split('_')[1]}"
        print(sample2)

        CT_df_re = CT_df2.drop(sample2, axis=1)
        
        ct_cols = CT_df_re.columns[CT_df_re.columns.str.startswith("CT")]
        ct_vals = CT_df_re[ct_cols].to_numpy()
        
        mi = np.mean(ct_vals, axis=1)
        SDi = np.std(ct_vals, axis=1)

        df = CT_df2.copy()
        small_IDs = [ind for ind in ca.df_allen["ID"] if ca.smallID_q(ind)]
        df = df[df["id"].isin(small_IDs)]

        sort_id = CT_df_re["id"].tolist()
        df = df.set_index("id").loc[sort_id].reset_index()

        vnorm = [(df[sample2].iloc[i] - mi.iloc[i]) / SDi.iloc[i] for i in range(len(df))]
        vnorm = np.nan_to_num(vnorm)
        vnorm = vnorm[vnorm != 0]
        CT_df_d = CT_df2.iloc[np.where(vnorm != 0)[0]]
        vnorm = np.array(vnorm)

        peakt = np.array(CT_df_d["LAG"])

        # Sensitivity
        corrs = []
        for b in bs:
            cos_p = np.sqrt(2) * np.cos(2 * np.pi * (peakt - b) / 24)
            corr, _ = pearsonr(vnorm, cos_p)
            corrs.append(corr)

        bc = bs[np.argmax(corrs)]
        cmax = np.max(corrs)

        ct_mod = int(sample.split("_")[0][2:]) % 24
        ct_key = str(ct_mod)

        if cmax > c_th_ave and ct_key in CT_c_c:
            CT_c_c[ct_key] += 1

        # Specificity
        vnorm_r = vnorm.copy()
        np.random.shuffle(vnorm_r)
        corrs_r = []
        for b in bs:
            cos_p = np.sqrt(2) * np.cos(2 * np.pi * (peakt - b) / 24)
            corr, _ = pearsonr(vnorm_r, cos_p)
            corrs_r.append(corr)
        cmax_r = np.max(corrs_r)

        if cmax_r < c_th_ave and ct_key in CT_r_c:
            CT_r_c[ct_key] += 1

CT_sens = [v / 24 * 100 for _, v in CT_c_c.items()]
CT_spec = [v / 24 * 100 for _, v in CT_r_c.items()]

print(f"FDR{FDR_th}, sensitivity:{CT_sens}%")
print(f"FDR{FDR_th}, specificity:{CT_spec}%")

In [ ]:
# Plot sensitivity and specificity by CT bins
fs = 15

plt.figure(figsize=(5, 5))
plt.plot(list(CT_c_c.keys()), CT_sens)
plt.scatter(list(CT_c_c.keys()), CT_sens)
plt.xlabel("CT (hour)", fontsize=fs)
plt.ylabel("Sensitivity (%)", fontsize=fs)
plt.ylim(0, 105)
plt.savefig(os.path.join(figdir, f"timetable_FDR{FDR_th}_sensitivity_byCT.SVG"))
plt.show()

plt.figure(figsize=(5, 5))
plt.plot(list(CT_c_c.keys()), CT_spec)
plt.scatter(list(CT_c_c.keys()), CT_spec)
plt.xlabel("CT (hour)", fontsize=fs)
plt.ylabel("Specificity (%)", fontsize=fs)
plt.ylim(0, 105)
plt.savefig(os.path.join(figdir, f"timetable_FDR{FDR_th}_specificity_byCT.SVG"))
plt.show()